In [4]:
import cv2
import time
import random
import cvzone
import subprocess
import threading
from cvzone.HandTrackingModule import HandDetector

In [5]:
# ---------------- POWERSHELL TTS ----------------
def speak_async(text):
    def run():
        command = f'''
        Add-Type -AssemblyName System.Speech;
        $speak = New-Object System.Speech.Synthesis.SpeechSynthesizer;
        $speak.Rate = 3;
        $speak.Speak("{text}");
        '''
        subprocess.run(["powershell", "-Command", command],
                       stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL)

    threading.Thread(target=run, daemon=True).start()

In [6]:

# ---------------- CAMERA ----------------
cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

detector = HandDetector(maxHands=1)

# ---------------- GAME VARIABLES ----------------
timer = 0
stateResult = False
startGame = False
initialTime = 0

scores = [0, 0]  # [AI, Player]
resultText = ""
imgAI = None

spokenResult = False
countdownStep = -1

# ---------------- MAIN LOOP ----------------
while True:
    imgBG = cv2.imread('Resources/BG.png')
    success, img = cap.read()

    if not success:
        break

    imgScaled = cv2.resize(img, None, fx=0.875, fy=0.875)
    imgScaled = imgScaled[:, 80:480]

    hands, imgScaled = detector.findHands(imgScaled)

    # ---------------- GAME START ----------------
    if startGame:

        if not stateResult:
            timer = time.time() - initialTime

            cv2.putText(imgBG, str(int(timer)), (605, 435),
                        cv2.FONT_HERSHEY_PLAIN, 6, (255, 0, 255), 4)

            # -------- COUNTDOWN VOICE --------
            if timer < 1 and countdownStep != 0:
                speak_async("Stone")
                countdownStep = 0

            elif 1 <= timer < 2 and countdownStep != 1:
                speak_async("Paper")
                countdownStep = 1

            elif 2 <= timer < 3 and countdownStep != 2:
                speak_async("Scissors")
                countdownStep = 2

            # -------- RESULT --------
            if timer > 3:
                stateResult = True
                timer = 0
                spokenResult = False

                randomNumber = random.randint(1, 3)
                imgAI = cv2.imread(f'Resources/{randomNumber}.png', cv2.IMREAD_UNCHANGED)

                playerMove = None

                if hands:
                    fingers = detector.fingersUp(hands[0])

                    if fingers == [0, 0, 0, 0, 0]:
                        playerMove = 1
                    elif fingers == [1, 1, 1, 1, 1]:
                        playerMove = 2
                    elif fingers == [0, 1, 1, 0, 0]:
                        playerMove = 3

                if playerMove is not None:

                    if (playerMove == 1 and randomNumber == 3) or \
                       (playerMove == 2 and randomNumber == 1) or \
                       (playerMove == 3 and randomNumber == 2):
                        scores[1] += 1
                        resultText = "YOUU WIN"

                    elif (playerMove == 3 and randomNumber == 1) or \
                         (playerMove == 1 and randomNumber == 2) or \
                         (playerMove == 2 and randomNumber == 3):
                        scores[0] += 1
                        resultText = "AI WINS"

                    else:
                        resultText = "DRAW"
                else:
                    resultText = "INVALID MOVE"

    # ---------------- UI ----------------
    imgBG[234:654, 795:1195] = imgScaled

    if stateResult and imgAI is not None:
        imgBG = cvzone.overlayPNG(imgBG, imgAI, (149, 310))

    # Scores
    cv2.putText(imgBG, str(scores[0]), (410, 215),
                cv2.FONT_HERSHEY_PLAIN, 4, (255, 255, 255), 6)

    cv2.putText(imgBG, str(scores[1]), (1112, 215),
                cv2.FONT_HERSHEY_PLAIN, 4, (255, 255, 255), 6)

    # Result + Voice
    if stateResult:
        cv2.putText(imgBG, resultText, (500, 100),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 0, 0), 4)

        if not spokenResult:
            speak_async(resultText.replace("YOU", "You").replace("AI", "A I"))
            spokenResult = True

    cv2.imshow("BG", imgBG)

    key = cv2.waitKey(1)

    # Start game
    if key == ord('s'):
        startGame = True
        stateResult = False
        initialTime = time.time()
        resultText = ""
        countdownStep = -1

    # Quit
    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()